# Gemini-3-Flash-Preview

In [4]:
# Cell 1: Setup imports, paths, and Gemini key (new google.genai SDK)
from pathlib import Path
import os
import re
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"
DESC_DIR = DATA_DIR / "descriptions"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl" / "gemini_3_flash"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")

# Load .env and configure Gemini client
load_dotenv(PROJECT_ROOT / ".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError("GOOGLE_API_KEY not found in .env")

client = genai.Client(api_key=GOOGLE_API_KEY)

# Gemini 3 Flash model code (preview)
MODEL_NAME = "gemini-3-flash-preview"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl/gemini_3_flash
VAL exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [5]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:03d}" for i in range(101, 201)]  # p101..p200

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 100
First/last: p101 p200
Prompt chars: 2544
Domain PDDL chars: 997


In [6]:
# Cell 3: Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()

In [7]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  nl_domain, nl_problem = load_problem_descriptions(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Create a syntactically correct and semantically consistent problem file "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the final problem PDDL.\n"
      "- Wrap it in exactly one <problem_file>...</problem_file> block.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not include any extra text before or after <problem_file>...</problem_file>."
  )

In [18]:
# Cell 5: Call Gemini for one prompt
def call_gemini(prompt: str, temperature: float = 0.4, max_output_tokens: int = 50000):
  response = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config={
          "temperature": temperature,
          "max_output_tokens": max_output_tokens,
      },
  )
  return (response.text or "").strip()

In [10]:
# Cell 6: Parse model output and extract problem_file block
def extract_problem_pddl(raw_text: str):
  if not raw_text:
      return ""

  matches = re.findall(r"<problem_file>(.*?)</problem_file>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if matches:
      return matches[-1].strip()

  return ""

In [12]:
# Cell 7: Solve one generated problem with live logs (planutils dual-bfws-ffparser)
import select

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  plan_path = work_dir / "plan"
  if plan_path.exists():
      plan_path.unlink()

  cmd = ["planutils", "run", "dual-bfws-ffparser", str(domain_pddl_path), str(problem_pddl_path)]
  print(f"[solver] start: {' '.join(cmd)}")
  print(f"[solver] timeout: {timeout_sec}s")

  proc = subprocess.Popen(
      cmd,
      cwd=work_dir,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
      bufsize=1,
  )

  stdout_lines = []
  stderr_lines = []
  start = time.time()

  # Live-stream both stdout and stderr while process runs
  while True:
      if proc.poll() is not None:
          break

      elapsed = time.time() - start
      if elapsed > timeout_sec:
          proc.kill()
          print(f"[solver] timeout reached ({timeout_sec}s), process killed.")
          break

      ready, _, _ = select.select([proc.stdout, proc.stderr], [], [], 0.2)
      for stream in ready:
          line = stream.readline()
          if not line:
              continue
          line = line.rstrip("\n")
          if stream is proc.stdout:
              stdout_lines.append(line)
              print(f"[solver][out] {line}")
          else:
              stderr_lines.append(line)
              print(f"[solver][err] {line}")

  # Drain remaining output
  out_rest, err_rest = proc.communicate(timeout=2)
  if out_rest:
      for line in out_rest.splitlines():
          stdout_lines.append(line)
          print(f"[solver][out] {line}")
  if err_rest:
      for line in err_rest.splitlines():
          stderr_lines.append(line)
          print(f"[solver][err] {line}")

  stdout = "\n".join(stdout_lines).strip()
  stderr = "\n".join(stderr_lines).strip()

  if plan_path.exists():
      plan_text = plan_path.read_text().strip()
      if plan_text:
          print("[solver] plan found.")
          return {
              "plan_found": True,
              "plan_text": plan_text,
              "solver_error": "",
              "stdout": stdout,
              "stderr": stderr,
          }

  if (time.time() - start) > timeout_sec and proc.returncode is None:
      solver_error = f"timeout after {timeout_sec}s"
  else:
      solver_error = (stderr + "\n" + stdout).strip() or "plan not found"

  print("[solver] no valid plan file produced.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": solver_error,
      "stdout": stdout,
      "stderr": stderr,
  }

In [13]:
# Cell 8: Validate plan with VAL (live logs, no timeout)
import select

def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  print(f"[val] start: {' '.join(cmd)}")

  proc = subprocess.Popen(
      cmd,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
      bufsize=1,
  )

  out_lines = []
  err_lines = []

  # Live-stream until process exits
  while True:
      if proc.poll() is not None:
          break

      ready, _, _ = select.select([proc.stdout, proc.stderr], [], [], 0.2)
      for stream in ready:
          line = stream.readline()
          if not line:
              continue
          line = line.rstrip("\n")
          if stream is proc.stdout:
              out_lines.append(line)
              print(f"[val][out] {line}")
          else:
              err_lines.append(line)
              print(f"[val][err] {line}")

  # Drain remaining output
  out_rest, err_rest = proc.communicate(timeout=2)
  if out_rest:
      for line in out_rest.splitlines():
          out_lines.append(line)
          print(f"[val][out] {line}")
  if err_rest:
      for line in err_rest.splitlines():
          err_lines.append(line)
          print(f"[val][err] {line}")

  out_text = "\n".join(out_lines).strip()
  err_text = "\n".join(err_lines).strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  print(f"[val] done. valid={val_ok}")
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [14]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, problem_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Generated problem PDDL
  (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  return cand_dir

In [15]:
# Cell 10: Run one problem through full pipeline
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_gemini(prompt)

  problem_pddl_text = extract_problem_pddl(raw_text)

  # If model fails to return a tagged problem file, record and stop early
  if not problem_pddl_text.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <problem_file>...</problem_file> found in model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_text, "", solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # Write generated problem first so solver can consume it
  problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
  problem_dir.mkdir(parents=True, exist_ok=True)
  gen_problem_path = problem_dir / "problem.pddl"
  gen_problem_path.write_text(problem_pddl_text)

  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=gen_problem_path,
      work_dir=problem_dir,
      timeout_sec=300,
  )

  val_result = None
  val_ok = False
  if solver_result["plan_found"]:
      plan_path = problem_dir / "plan"
      if not plan_path.exists():
          # fallback safety if solver wrote plan text but not file
          plan_path = problem_dir / "plan.txt"
          plan_path.write_text(solver_result["plan_text"])
      val_result = run_val(DOMAIN_PDDL_PATH, gen_problem_path, plan_path)
      val_ok = bool(val_result["val_ok"])

  # Persist final artifacts
  save_problem_outputs(problem_id, raw_text, problem_pddl_text, solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result["plan_found"]),
      "val_ok": val_ok,
      "solver_error": solver_result.get("solver_error", ""),
  }

In [ ]:
# Cell 11: Generate only (no solver/VAL)
smoke_problem = "p101"

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
problem_pddl_text = extract_problem_pddl(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_problem_pddl_chars:", len(problem_pddl_text or ""))
print("saved:", cand_dir)

In [ ]:
# Cell 12: Solver+VAL only (no model call)
smoke_problem = "p101"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

problem_path = cand_dir / "problem.pddl"
if not problem_path.exists() or not problem_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty generated problem file: {problem_path}")

solver_result = run_solver(
  domain_pddl_path=DOMAIN_PDDL_PATH,
  problem_pddl_path=problem_path,
  work_dir=cand_dir,
  timeout_sec=300,
)

# Save solver logs/results
(cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
(cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

val_result = None
if solver_result["plan_found"]:
  # solver writes ./plan; keep a canonical copy as plan.txt
  plan_file = cand_dir / "plan"
  plan_text = solver_result.get("plan_text", "")
  (cand_dir / "plan.txt").write_text(plan_text)

  val_result = run_val(DOMAIN_PDDL_PATH, problem_path, plan_file)
  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
else:
  (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

summary = {
  "problem": smoke_problem,
  "plan_found": bool(solver_result["plan_found"]),
  "val_ok": bool(val_result["val_ok"]) if val_result is not None else False,
  "solver_error": solver_result.get("solver_error", ""),
}
print(json.dumps(summary, indent=2))

# Diagnosing cells

In [20]:
import os
os.environ["PATH"] = f"/home/sagemaker-user/.local/bin:{os.environ['PATH']}"
print(os.environ["PATH"])

/home/sagemaker-user/.local/bin:/opt/conda/bin:/opt/conda/condabin:/etc/sagemaker-inference-server:/opt/conda/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin


In [21]:
!which planutils
!planutils --help

/home/sagemaker-user/.local/bin/planutils
usage: planutils [-h]
                 {install,uninstall,run,remote,remote-list,check-installed,server,list,setup,upgrade,configure,show}
                 ...

positional arguments:
  {install,uninstall,run,remote,remote-list,check-installed,server,list,setup,upgrade,configure,show}
                        sub-command help
    install             install package(s) such as a planner
    uninstall           uninstall package(s)
    run                 run package
    remote              run package remotely
    remote-list         list the available remote packages
    check-installed     check if a package is installed
    server              start a server for runnable packages
    list                list the available packages
    setup               setup planutils for current user
    upgrade             upgrade all of the installed packages
    configure           configure planutils
    show                show details about a particula

In [17]:
# Quick test cell: inspect Gemini finish reason for one prompt
test_prompt = build_prompt("p101")

resp = client.models.generate_content(
  model=MODEL_NAME,
  contents=test_prompt,
  config={
      "temperature": 0.4,
      "max_output_tokens": 10000,
  },
)

print("Response text length:", len(resp.text or ""))
print("Response text preview:\n", (resp.text or "")[:500])

# Try common locations for finish reason in SDK response object
try:
  cands = getattr(resp, "candidates", None) or []
  if cands:
      print("finish_reason:", getattr(cands[0], "finish_reason", None))
      print("safety_ratings:", getattr(cands[0], "safety_ratings", None))
  else:
      print("No candidates field found.")
except Exception as e:
  print("Could not read candidates:", e)

# Optional raw dump for debugging
print("\nRaw response object:")
print(resp)

Response text length: 963
Response text preview:
 <problem_file>
(define
	(problem blocksworld_55)
	(:domain blocksworld)
	(:objects
		block1 block2 block3 block4 block5 block6 block7 block8 block9 block10 block11 block12 block13 block14 block15 block16 block17 block18 block19 block20 block21 block22 block23 block24 block25 block26 block27 block28 block29 block30 block31 block32 block33 block34 block35 block36 block37 block38 block39 block40 block41 block42 block43 block44 block45 block46 block47 block48 block49 block50 block51 block52 block53 
finish_reason: FinishReason.MAX_TOKENS
safety_ratings: None

Raw response object:
sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  citation_metadata=CitationMetadata(
    citations=[
      Citation(
        end_index=518,
        license='',
        start_index=180,
        uri='https://stackoverflow.com/questions/61443414/how-to-solve-programming-chess-piece-logical-problem'
      ),
    ]
  ),
  content=Conten